# 03 - Preprocessing and Modeling-Ready Dataset Construction

This notebook turns the raw entity-resolved rainfall table into one compact processed dataset for later modeling or benchmark construction.

It follows the EDA conclusion from notebook 02:

- Do not blindly average NASA POWER and Open-Meteo into a single final target.
- Preserve source-specific rainfall columns.
- Create uncertainty features from cross-source disagreement.
- Build multiple target candidates for sensitivity analysis.
- Apply city-month bias calibration using training-period statistics only.
- Use temporal train/validation/test split labels to avoid leakage.
- Save only the main processed dataset in `data/processed/`.
- Save all audit/report tables in `reports/03_preprocessing/`.

## Methodological Decisions

The raw table has no missing values in source measurements or rainfall targets. Therefore, raw target imputation is skipped.

Missingness is introduced later by forecasting-safe temporal features:

- `rainfall_lag_1d_mm`, `rainfall_lag_7d_mm`, and `rainfall_lag_30d_mm` need past observations.
- rolling features need enough prior-day history.
- previous wet/dry spell features are undefined on the first date of each city.

For those engineered predictor columns only, imputation uses training data statistics:

$$
\tilde{x}_{i,t} =
\begin{cases}
x_{i,t}, & \text{if observed} \\
\mathrm{median}_{train}(x \mid entity_i), & \text{if missing and entity median exists} \\
\mathrm{median}_{train}(x), & \text{fallback}
\end{cases}
$$

Each imputed predictor also gets a binary `_was_imputed` indicator.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import warnings

import numpy as np
import pandas as pd

try:
    display
except NameError:
    def display(obj):
        if hasattr(obj, 'to_string'):
            print(obj.to_string())
        else:
            print(obj)

warnings.filterwarnings('ignore', category=FutureWarning)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports' / '03_preprocessing'
TABLE_DIR = REPORT_DIR / 'tables'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

BASE_STEM = 'sea_rainfall_daily_2020_2025'
RESOLVED_PATH = RAW_DIR / f'{BASE_STEM}_entity_resolved.csv'
REGISTRY_PATH = RAW_DIR / f'{BASE_STEM}_entity_registry.csv'
MANIFEST_RAW_PATH = RAW_DIR / f'{BASE_STEM}_manifest.json'

# Keep data/processed compact: one modeling-ready table with a split column.
PROCESSED_MAIN_PATH = PROCESSED_DIR / f'{BASE_STEM}_processed_features.csv'

# Keep documentation and audit artifacts in reports/03_preprocessing.
BIAS_CORRECTION_PATH = TABLE_DIR / '01_city_month_bias_correction.csv'
IMPUTATION_REPORT_PATH = TABLE_DIR / '02_imputation_report.csv'
SCALING_PARAMETERS_PATH = TABLE_DIR / '03_scaling_parameters.csv'
FEATURE_DICTIONARY_PATH = TABLE_DIR / '04_processed_feature_dictionary.csv'
QUALITY_REPORT_PATH = TABLE_DIR / '05_processed_quality_report.csv'
SPLIT_SUMMARY_PATH = TABLE_DIR / '06_temporal_split_summary.csv'
README_PATH = REPORT_DIR / 'PREPROCESSING_README.md'
MANIFEST_PATH = REPORT_DIR / f'{BASE_STEM}_processed_manifest.json'

MONTH_LABELS = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw input: {RESOLVED_PATH}')
print(f'Processed data output: {PROCESSED_MAIN_PATH}')
print(f'Preprocessing report directory: {REPORT_DIR}')

Project root: D:\DS\Project
Raw input: D:\DS\Project\data\raw\sea_rainfall_daily_2020_2025_entity_resolved.csv
Processed data output: D:\DS\Project\data\processed\sea_rainfall_daily_2020_2025_processed_features.csv
Preprocessing report directory: D:\DS\Project\reports\03_preprocessing


## 1. Load Raw Entity-Resolved Data

The input table is the output of notebook 01 after entity resolution. Each row is one canonical city-date pair with NASA POWER and Open-Meteo measurements side by side.

In [2]:
raw_df = pd.read_csv(RESOLVED_PATH, parse_dates=['date'])
registry_df = pd.read_csv(REGISTRY_PATH)

with MANIFEST_RAW_PATH.open('r', encoding='utf-8') as file:
    raw_manifest = json.load(file)

raw_df = raw_df.sort_values(['entity_id', 'date']).reset_index(drop=True)

print('Raw entity-resolved shape:', raw_df.shape)
print('Date range:', raw_df['date'].min().date(), 'to', raw_df['date'].max().date())
print('Entities:', raw_df['entity_id'].nunique())
display(raw_df.head())

Raw entity-resolved shape: (26304, 22)
Date range: 2020-01-01 to 2025-12-31
Entities: 12


,entity_id,date,country,location_name,canonical_latitude,canonical_longitude,nasa_power_precipitation_mm,nasa_power_temp_mean_c,nasa_power_temp_max_c,nasa_power_temp_min_c,...,nasa_power_surface_pressure_kpa,open_meteo_precipitation_mm,open_meteo_temp_mean_c,open_meteo_temp_max_c,open_meteo_temp_min_c,open_meteo_relative_humidity_pct,open_meteo_wind_speed_ms,open_meteo_surface_pressure_kpa,precipitation_mm_mean_two_sources,precipitation_mm_abs_diff
0,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-01,Brunei,Bandar Seri Begawan,4.9031,114.9398,2.29,27.59,29.24,26.03,...,100.70,1.1,27.1,30.0,23.7,85.0,2.5833,101.07,1.695,1.19
1,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-02,Brunei,Bandar Seri Begawan,4.9031,114.9398,1.17,27.46,29.20,25.72,...,100.79,1.4,26.9,29.4,24.7,87.0,2.1944,101.09,1.285,0.23
2,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-03,Brunei,Bandar Seri Begawan,4.9031,114.9398,4.84,27.33,28.77,26.13,...,100.68,1.9,26.7,28.8,24.6,88.0,2.0556,100.96,3.370,2.94
3,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-04,Brunei,Bandar Seri Begawan,4.9031,114.9398,2.77,27.18,28.73,25.57,...,100.53,0.5,27.2,29.2,25.0,85.0,3.3889,100.76,1.635,2.27
4,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-05,Brunei,Bandar Seri Begawan,4.9031,114.9398,0.46,27.45,29.45,25.61,...,100.51,8.6,26.8,30.0,24.3,87.0,1.5833,100.81,4.530,8.14


## 2. Quality Gate Before Preprocessing

This gate decides whether cleaning or target imputation is needed. The raw table is expected to have:

- no duplicated `entity_id + date` keys;
- no negative rainfall values;
- no missing source measurements.

In [3]:
key_duplicate_rows = int(raw_df.duplicated(['entity_id', 'date']).sum())
rainfall_columns = [
    'nasa_power_precipitation_mm',
    'open_meteo_precipitation_mm',
    'precipitation_mm_mean_two_sources',
]
negative_rainfall_rows = int((raw_df[rainfall_columns] < 0).any(axis=1).sum())
raw_missing_counts = raw_df.isna().sum()
target_missing_rows = int(raw_df[rainfall_columns].isna().any(axis=1).sum())

preprocessing_gate = pd.DataFrame([
    {'check': 'duplicate_entity_date_rows', 'value': key_duplicate_rows, 'expected': 0, 'passed': key_duplicate_rows == 0},
    {'check': 'negative_rainfall_rows', 'value': negative_rainfall_rows, 'expected': 0, 'passed': negative_rainfall_rows == 0},
    {'check': 'target_missing_rows', 'value': target_missing_rows, 'expected': 0, 'passed': target_missing_rows == 0},
    {'check': 'raw_missing_cells', 'value': int(raw_missing_counts.sum()), 'expected': 0, 'passed': int(raw_missing_counts.sum()) == 0},
])

if not preprocessing_gate['passed'].all():
    raise ValueError('Preprocessing gate failed. Inspect preprocessing_gate before continuing.')

display(preprocessing_gate)

,check,value,expected,passed
0,duplicate_entity_date_rows,0,0,True
1,negative_rainfall_rows,0,0,True
2,target_missing_rows,0,0,True
3,raw_missing_cells,0,0,True


## 3. Temporal Splits

The project is framed as rainfall forecasting/data construction, so the split must respect time.

- Train: 2020-01-01 to 2023-12-31
- Validation: 2024-01-01 to 2024-12-31
- Test: 2025-01-01 to 2025-12-31

All calibration, scaling, and imputation statistics are learned from the training period only.

In [4]:
def assign_temporal_split(date):
    if date < pd.Timestamp('2024-01-01'):
        return 'train'
    if date < pd.Timestamp('2025-01-01'):
        return 'validation'
    return 'test'


df = raw_df.copy()
df['split'] = df['date'].map(assign_temporal_split)
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['month_name'] = df['month'].map(lambda month: MONTH_LABELS[month - 1])
df['day_of_year'] = df['date'].dt.dayofyear
df['is_leap_year'] = df['date'].dt.is_leap_year.astype(int)
df['season_quarter'] = df['month'].map({
    12: 'DJF', 1: 'DJF', 2: 'DJF',
    3: 'MAM', 4: 'MAM', 5: 'MAM',
    6: 'JJA', 7: 'JJA', 8: 'JJA',
    9: 'SON', 10: 'SON', 11: 'SON',
})
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 366)
df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 366)

split_summary = (
    df.groupby('split', as_index=False)
    .agg(
        rows=('entity_id', 'size'),
        start_date=('date', 'min'),
        end_date=('date', 'max'),
        entities=('entity_id', 'nunique'),
    )
)
display(split_summary)

,split,rows,start_date,end_date,entities
0,test,4380,2025-01-01,2025-12-31,12
1,train,17532,2020-01-01,2023-12-31,12
2,validation,4392,2024-01-01,2024-12-31,12


## 4. Target Candidates and Source-Uncertainty Features

Because notebook 02 found weak-to-moderate agreement between sources, this notebook creates several target candidates instead of selecting a single final ground truth.

City-month source bias is estimated only from the training period:

$$
b_{i,m} = mean_{train}(NASA_{i,t} - OpenMeteo_{i,t} \mid entity=i,\ month=m)
$$

Open-Meteo calibrated toward NASA:

$$
OpenMeteo^{NASAref}_{i,t} = max(OpenMeteo_{i,t} + b_{i,m}, 0)
$$

NASA calibrated toward Open-Meteo:

$$
NASA^{Openref}_{i,t} = max(NASA_{i,t} - b_{i,m}, 0)
$$

Consensus target candidates:

$$
target^{NASAref}_{i,t} = \frac{NASA_{i,t} + OpenMeteo^{NASAref}_{i,t}}{2}
$$

$$
target^{Openref}_{i,t} = \frac{NASA^{Openref}_{i,t} + OpenMeteo_{i,t}}{2}
$$

These are not declared as true rainfall. They are target candidates for sensitivity analysis.

In [5]:
df['target_nasa_power_precipitation_mm'] = df['nasa_power_precipitation_mm']
df['target_open_meteo_precipitation_mm'] = df['open_meteo_precipitation_mm']
df['target_baseline_two_source_mean_mm'] = df['precipitation_mm_mean_two_sources']
df['source_bias_nasa_minus_open_meteo_mm'] = df['nasa_power_precipitation_mm'] - df['open_meteo_precipitation_mm']
df['source_abs_diff_precipitation_mm'] = df['source_bias_nasa_minus_open_meteo_mm'].abs()
df['source_relative_abs_diff_precipitation'] = df['source_abs_diff_precipitation_mm'] / (df['target_baseline_two_source_mean_mm'] + 1.0)
df['nasa_wet_day'] = (df['nasa_power_precipitation_mm'] >= 1.0).astype(int)
df['open_meteo_wet_day'] = (df['open_meteo_precipitation_mm'] >= 1.0).astype(int)
df['wet_day_disagreement'] = (df['nasa_wet_day'] != df['open_meteo_wet_day']).astype(int)
df['high_source_gap_10mm'] = (df['source_abs_diff_precipitation_mm'] > 10).astype(int)
df['high_source_gap_20mm'] = (df['source_abs_diff_precipitation_mm'] > 20).astype(int)

train_mask = df['split'] == 'train'
train_bias_city_month = (
    df.loc[train_mask]
    .groupby(['entity_id', 'month'], as_index=False)
    .agg(
        source_bias_train_city_month_mm=('source_bias_nasa_minus_open_meteo_mm', 'mean'),
        source_bias_train_city_month_std_mm=('source_bias_nasa_minus_open_meteo_mm', 'std'),
        source_bias_train_city_month_n=('source_bias_nasa_minus_open_meteo_mm', 'size'),
    )
)
train_bias_city = (
    df.loc[train_mask]
    .groupby('entity_id', as_index=False)
    .agg(source_bias_train_city_fallback_mm=('source_bias_nasa_minus_open_meteo_mm', 'mean'))
)
global_train_bias = float(df.loc[train_mask, 'source_bias_nasa_minus_open_meteo_mm'].mean())

bias_table = train_bias_city_month.merge(train_bias_city, on='entity_id', how='left')
bias_table['source_bias_train_global_fallback_mm'] = global_train_bias
bias_table['bias_formula'] = 'mean_train(nasa_power_precipitation_mm - open_meteo_precipitation_mm | entity_id, month)'

df = df.merge(
    bias_table[['entity_id', 'month', 'source_bias_train_city_month_mm', 'source_bias_train_city_fallback_mm']],
    on=['entity_id', 'month'],
    how='left',
)
df['source_bias_train_city_month_mm'] = (
    df['source_bias_train_city_month_mm']
    .fillna(df['source_bias_train_city_fallback_mm'])
    .fillna(global_train_bias)
)

df['open_meteo_calibrated_to_nasa_mm'] = (
    df['open_meteo_precipitation_mm'] + df['source_bias_train_city_month_mm']
).clip(lower=0)
df['nasa_power_calibrated_to_open_meteo_mm'] = (
    df['nasa_power_precipitation_mm'] - df['source_bias_train_city_month_mm']
).clip(lower=0)
df['target_nasa_reference_consensus_mm'] = (
    df['nasa_power_precipitation_mm'] + df['open_meteo_calibrated_to_nasa_mm']
) / 2
df['target_open_meteo_reference_consensus_mm'] = (
    df['nasa_power_calibrated_to_open_meteo_mm'] + df['open_meteo_precipitation_mm']
) / 2
df['target_reference_consensus_band_mm'] = (
    df['target_nasa_reference_consensus_mm'] - df['target_open_meteo_reference_consensus_mm']
).abs()

display(bias_table.head())
display(df[[
    'entity_id', 'date', 'split',
    'target_nasa_power_precipitation_mm',
    'target_open_meteo_precipitation_mm',
    'target_baseline_two_source_mean_mm',
    'source_bias_train_city_month_mm',
    'target_nasa_reference_consensus_mm',
    'target_open_meteo_reference_consensus_mm',
    'target_reference_consensus_band_mm',
]].head())

,entity_id,month,source_bias_train_city_month_mm,source_bias_train_city_month_std_mm,source_bias_train_city_month_n,source_bias_train_city_fallback_mm,source_bias_train_global_fallback_mm,bias_formula
0,SEA_BN_BANDAR_SERI_BEGAWAN,1,-0.436290,12.292484,124,-0.402444,-0.220883,mean_train(nasa_power_precipitation_mm - open_...
1,SEA_BN_BANDAR_SERI_BEGAWAN,2,0.280354,14.824378,113,-0.402444,-0.220883,mean_train(nasa_power_precipitation_mm - open_...
2,SEA_BN_BANDAR_SERI_BEGAWAN,3,-0.364677,14.562312,124,-0.402444,-0.220883,mean_train(nasa_power_precipitation_mm - open_...
3,SEA_BN_BANDAR_SERI_BEGAWAN,4,-1.250750,10.788342,120,-0.402444,-0.220883,mean_train(nasa_power_precipitation_mm - open_...
4,SEA_BN_BANDAR_SERI_BEGAWAN,5,-0.266532,9.671658,124,-0.402444,-0.220883,mean_train(nasa_power_precipitation_mm - open_...


,entity_id,date,split,target_nasa_power_precipitation_mm,target_open_meteo_precipitation_mm,target_baseline_two_source_mean_mm,source_bias_train_city_month_mm,target_nasa_reference_consensus_mm,target_open_meteo_reference_consensus_mm,target_reference_consensus_band_mm
0,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-01,train,2.29,1.1,1.695,-0.43629,1.476855,1.913145,0.43629
1,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-02,train,1.17,1.4,1.285,-0.43629,1.066855,1.503145,0.43629
2,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-03,train,4.84,1.9,3.370,-0.43629,3.151855,3.588145,0.43629
3,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-04,train,2.77,0.5,1.635,-0.43629,1.416855,1.853145,0.43629
4,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-05,train,0.46,8.6,4.530,-0.43629,4.311855,4.748145,0.43629


## 5. Meteorological Predictors and Forecasting-Safe History Features

Same-day source measurements are kept as candidate predictors for benchmark construction. For strict forecasting settings, downstream modeling should prefer lagged or externally available forecast variables.

Rainfall history features are shifted by one day, so they use information available before the target date.

In [6]:
for variable in ['temp_mean_c', 'temp_max_c', 'temp_min_c', 'relative_humidity_pct', 'wind_speed_ms', 'surface_pressure_kpa']:
    nasa_col = f'nasa_power_{variable}'
    open_col = f'open_meteo_{variable}'
    df[f'{variable}_mean_two_sources'] = df[[nasa_col, open_col]].mean(axis=1)
    df[f'{variable}_source_diff_nasa_minus_open_meteo'] = df[nasa_col] - df[open_col]
    df[f'{variable}_source_abs_diff'] = df[f'{variable}_source_diff_nasa_minus_open_meteo'].abs()


def add_forecasting_history_features(group):
    group = group.sort_values('date').copy()
    rainfall = group['target_baseline_two_source_mean_mm']
    prior_rainfall = rainfall.shift(1)

    group['rainfall_lag_1d_mm'] = rainfall.shift(1)
    group['rainfall_lag_7d_mm'] = rainfall.shift(7)
    group['rainfall_lag_30d_mm'] = rainfall.shift(30)
    group['rainfall_rolling_7d_mean_prev_mm'] = prior_rainfall.rolling(7, min_periods=3).mean()
    group['rainfall_rolling_30d_sum_prev_mm'] = prior_rainfall.rolling(30, min_periods=10).sum()
    group['rainfall_rolling_90d_mean_prev_mm'] = prior_rainfall.rolling(90, min_periods=30).mean()

    wet_current = rainfall >= 1.0
    dry_current = ~wet_current
    wet_groups = (wet_current != wet_current.shift()).cumsum()
    dry_groups = (dry_current != dry_current.shift()).cumsum()

    wet_spell_current = wet_current.groupby(wet_groups).cumcount() + 1
    wet_spell_current = wet_spell_current.where(wet_current, 0)
    dry_spell_current = dry_current.groupby(dry_groups).cumcount() + 1
    dry_spell_current = dry_spell_current.where(dry_current, 0)

    group['wet_spell_days_prev'] = wet_spell_current.shift(1)
    group['dry_spell_days_prev'] = dry_spell_current.shift(1)
    group['was_wet_previous_day'] = wet_current.shift(1).astype('float')
    return group


df = (
    df.groupby('entity_id', group_keys=False)
    .apply(add_forecasting_history_features)
    .reset_index(drop=True)
)

# Recompute the mask after groupby/apply resets the row index.
train_mask = df['split'] == 'train'

history_cols = [
    'rainfall_lag_1d_mm',
    'rainfall_lag_7d_mm',
    'rainfall_lag_30d_mm',
    'rainfall_rolling_7d_mean_prev_mm',
    'rainfall_rolling_30d_sum_prev_mm',
    'rainfall_rolling_90d_mean_prev_mm',
    'wet_spell_days_prev',
    'dry_spell_days_prev',
    'was_wet_previous_day',
]

display(df[['entity_id', 'date'] + history_cols].head(35))

,entity_id,date,rainfall_lag_1d_mm,rainfall_lag_7d_mm,rainfall_lag_30d_mm,rainfall_rolling_7d_mean_prev_mm,rainfall_rolling_30d_sum_prev_mm,rainfall_rolling_90d_mean_prev_mm,wet_spell_days_prev,dry_spell_days_prev,was_wet_previous_day
0,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-02,1.695,NaN,NaN,NaN,NaN,NaN,1.0,0.0,1.0
2,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-03,1.285,NaN,NaN,NaN,NaN,NaN,2.0,0.0,1.0
3,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-04,3.370,NaN,NaN,2.116667,NaN,NaN,3.0,0.0,1.0
4,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-05,1.635,NaN,NaN,1.996250,NaN,NaN,4.0,0.0,1.0
5,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-06,4.530,NaN,NaN,2.503000,NaN,NaN,5.0,0.0,1.0
6,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-07,0.640,NaN,NaN,2.192500,NaN,NaN,0.0,1.0,0.0
7,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-08,1.190,1.695,NaN,2.049286,NaN,NaN,1.0,0.0,1.0
8,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-09,39.310,1.285,NaN,7.422857,NaN,NaN,2.0,0.0,1.0
9,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-10,0.730,3.370,NaN,7.343571,NaN,NaN,0.0,1.0,0.0


## 6. Train-Only Imputation for Engineered History Features

No raw source measurement or target column is imputed.

Only engineered history predictors are imputed because the first dates of each entity have no previous observations. This is structural missingness caused by feature construction, not sensor failure.

In [7]:
imputation_rows = []
for column in history_cols:
    missing_before = int(df[column].isna().sum())
    df[f'{column}_was_imputed'] = df[column].isna().astype(int)

    train_medians_by_entity = df.loc[train_mask].groupby('entity_id')[column].median()
    global_train_median = float(df.loc[train_mask, column].median())
    fill_values = df['entity_id'].map(train_medians_by_entity).fillna(global_train_median)
    df[column] = df[column].fillna(fill_values)
    missing_after = int(df[column].isna().sum())

    imputation_rows.append({
        'column_name': column,
        'missing_before': missing_before,
        'missing_after': missing_after,
        'imputation_applied': missing_before > 0,
        'global_train_median_fallback': global_train_median,
        'formula': 'if missing, fill with median_train(column | entity_id); fallback median_train(column)',
        'reason': 'Structural warm-up missingness from lag/rolling/spell feature construction.',
    })

imputation_report = pd.DataFrame(imputation_rows)

target_columns = [
    'target_nasa_power_precipitation_mm',
    'target_open_meteo_precipitation_mm',
    'target_baseline_two_source_mean_mm',
    'target_nasa_reference_consensus_mm',
    'target_open_meteo_reference_consensus_mm',
]
target_missing_after = int(df[target_columns].isna().any(axis=1).sum())
if target_missing_after:
    raise ValueError(f'Target columns still contain missing values: {target_missing_after} rows')

display(imputation_report)

,column_name,missing_before,missing_after,imputation_applied,global_train_median_fallback,formula,reason
0,rainfall_lag_1d_mm,12,0,True,3.470000,"if missing, fill with median_train(column | en...",Structural warm-up missingness from lag/rollin...
1,rainfall_lag_7d_mm,84,0,True,3.475000,"if missing, fill with median_train(column | en...",Structural warm-up missingness from lag/rollin...
2,rainfall_lag_30d_mm,360,0,True,3.520000,"if missing, fill with median_train(column | en...",Structural warm-up missingness from lag/rollin...
3,rainfall_rolling_7d_mean_prev_mm,36,0,True,5.582857,"if missing, fill with median_train(column | en...",Structural warm-up missingness from lag/rollin...
4,rainfall_rolling_30d_sum_prev_mm,120,0,True,193.890000,"if missing, fill with median_train(column | en...",Structural warm-up missingness from lag/rollin...
5,rainfall_rolling_90d_mean_prev_mm,360,0,True,7.019944,"if missing, fill with median_train(column | en...",Structural warm-up missingness from lag/rollin...
6,wet_spell_days_prev,12,0,True,4.000000,"if missing, fill with median_train(column | en...",Structural warm-up missingness from lag/rollin...
7,dry_spell_days_prev,12,0,True,0.000000,"if missing, fill with median_train(column | en...",Structural warm-up missingness from lag/rollin...
8,was_wet_previous_day,12,0,True,1.000000,"if missing, fill with median_train(column | en...",Structural warm-up missingness from lag/rollin...


## 7. Train-Only Scaling Parameters

Continuous predictor scaling uses training-period statistics only:

\[
z = \frac{x - \mu_{train}}{\sigma_{train}}
\]

The original physical-unit columns remain in the processed table. Scaled columns use the `z_` prefix.

In [8]:
continuous_predictor_cols = [
    'canonical_latitude', 'canonical_longitude',
    'month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos',
    'source_abs_diff_precipitation_mm',
    'source_relative_abs_diff_precipitation',
    'source_bias_train_city_month_mm',
    'target_reference_consensus_band_mm',
    'temp_mean_c_mean_two_sources',
    'temp_max_c_mean_two_sources',
    'temp_min_c_mean_two_sources',
    'relative_humidity_pct_mean_two_sources',
    'wind_speed_ms_mean_two_sources',
    'surface_pressure_kpa_mean_two_sources',
    'temp_mean_c_source_abs_diff',
    'relative_humidity_pct_source_abs_diff',
    'wind_speed_ms_source_abs_diff',
    'surface_pressure_kpa_source_abs_diff',
] + history_cols

scaling_rows = []
for column in continuous_predictor_cols:
    train_values = df.loc[train_mask, column]
    mean_value = float(train_values.mean())
    std_value = float(train_values.std(ddof=0))
    if not np.isfinite(std_value) or std_value == 0:
        std_value = 1.0
    df[f'z_{column}'] = (df[column] - mean_value) / std_value
    scaling_rows.append({
        'column_name': column,
        'scaled_column_name': f'z_{column}',
        'train_mean': mean_value,
        'train_std': std_value,
        'formula': 'z = (x - train_mean) / train_std',
    })

scaling_parameters = pd.DataFrame(scaling_rows)
display(scaling_parameters.head())

,column_name,scaled_column_name,train_mean,train_std,formula
0,canonical_latitude,z_canonical_latitude,8.434017e+00,9.109890,z = (x - train_mean) / train_std
1,canonical_longitude,z_canonical_longitude,1.075448e+02,8.277412,z = (x - train_mean) / train_std
2,month_sin,z_month_sin,-4.784655e-03,0.705758,z = (x - train_mean) / train_std
3,month_cos,z_month_cos,-2.028817e-03,0.708434,z = (x - train_mean) / train_std
4,day_of_year_sin,z_day_of_year_sin,-1.621133e-18,0.707832,z = (x - train_mean) / train_std


## 8. Assemble Slim Processed Dataset

The intermediate dataframe contains many audit columns. The exported processed dataset should be smaller and safer for modeling.

Keep in `data/processed`:

- identifiers, location metadata, and temporal split;
- target candidates;
- compact uncertainty features;
- compact weather summaries;
- forecasting-safe lag/rolling/spell features;
- imputation indicators.

Drop from the main processed dataset:

- duplicated raw source rainfall columns, because they are already represented as target candidates;
- source-specific raw weather columns, because the table keeps two-source mean and source-gap summaries instead;
- `z_` scaled columns, because scaling parameters are saved in the preprocessing report and can be applied later inside training pipelines;
- intermediate calibration columns.

The raw side-by-side source measurements remain available in `data/raw/`, and audit tables remain in `reports/03_preprocessing/`.

In [9]:
id_cols = [
    'entity_id', 'date', 'split', 'country', 'location_name',
    'canonical_latitude', 'canonical_longitude',
    'year', 'month', 'month_name', 'day_of_year', 'is_leap_year', 'season_quarter',
]

target_candidate_cols = [
    'target_nasa_power_precipitation_mm',
    'target_open_meteo_precipitation_mm',
    'target_baseline_two_source_mean_mm',
    'target_nasa_reference_consensus_mm',
    'target_open_meteo_reference_consensus_mm',
]

target_uncertainty_cols = [
    'target_reference_consensus_band_mm',
    'source_bias_nasa_minus_open_meteo_mm',
    'source_abs_diff_precipitation_mm',
    'source_relative_abs_diff_precipitation',
    'source_bias_train_city_month_mm',
    'wet_day_disagreement',
    'high_source_gap_10mm',
    'high_source_gap_20mm',
]

calendar_cols = [
    'month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos',
]

weather_mean_cols = [
    'temp_mean_c_mean_two_sources',
    'temp_max_c_mean_two_sources',
    'temp_min_c_mean_two_sources',
    'relative_humidity_pct_mean_two_sources',
    'wind_speed_ms_mean_two_sources',
    'surface_pressure_kpa_mean_two_sources',
]

weather_uncertainty_cols = [
    'temp_mean_c_source_abs_diff',
    'temp_max_c_source_abs_diff',
    'temp_min_c_source_abs_diff',
    'relative_humidity_pct_source_abs_diff',
    'wind_speed_ms_source_abs_diff',
    'surface_pressure_kpa_source_abs_diff',
]

imputation_indicator_cols = [f'{column}_was_imputed' for column in history_cols]

# Scaling parameters are saved to reports, but z_ columns are not exported to avoid doubling the feature count.
scaled_cols = [f'z_{column}' for column in continuous_predictor_cols]

processed_columns = (
    id_cols
    + target_candidate_cols
    + target_uncertainty_cols
    + calendar_cols
    + weather_mean_cols
    + weather_uncertainty_cols
    + history_cols
    + imputation_indicator_cols
)

processed_df = df.loc[:, list(dict.fromkeys(processed_columns))].copy()
processed_df = processed_df.sort_values(['entity_id', 'date']).reset_index(drop=True)

excluded_columns = sorted(set(df.columns) - set(processed_df.columns))
excluded_columns_report = pd.DataFrame([
    {
        'column_name': column,
        'reason': (
            'not_exported_to_keep_processed_dataset_slim; retained in raw data, intermediate notebook state, or preprocessing report'
        ),
    }
    for column in excluded_columns
])

processed_missing_cells = int(processed_df.isna().sum().sum())
if processed_missing_cells:
    missing_preview = processed_df.isna().sum().sort_values(ascending=False).head(20)
    raise ValueError(f'Processed dataset still has missing cells. Top missing columns:\n{missing_preview}')

processed_negative_targets = int((processed_df[target_candidate_cols] < 0).any(axis=1).sum())
if processed_negative_targets:
    raise ValueError(f'Negative target candidates detected: {processed_negative_targets}')

display(processed_df.head())
print('Processed shape:', processed_df.shape)
print('Excluded columns:', len(excluded_columns_report))

,entity_id,date,split,country,location_name,canonical_latitude,canonical_longitude,year,month,month_name,...,was_wet_previous_day,rainfall_lag_1d_mm_was_imputed,rainfall_lag_7d_mm_was_imputed,rainfall_lag_30d_mm_was_imputed,rainfall_rolling_7d_mean_prev_mm_was_imputed,rainfall_rolling_30d_sum_prev_mm_was_imputed,rainfall_rolling_90d_mean_prev_mm_was_imputed,wet_spell_days_prev_was_imputed,dry_spell_days_prev_was_imputed,was_wet_previous_day_was_imputed
0,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-01,train,Brunei,Bandar Seri Begawan,4.9031,114.9398,2020,1,Jan,...,1.0,1,1,1,1,1,1,1,1,1
1,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-02,train,Brunei,Bandar Seri Begawan,4.9031,114.9398,2020,1,Jan,...,1.0,0,1,1,1,1,1,0,0,0
2,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-03,train,Brunei,Bandar Seri Begawan,4.9031,114.9398,2020,1,Jan,...,1.0,0,1,1,1,1,1,0,0,0
3,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-04,train,Brunei,Bandar Seri Begawan,4.9031,114.9398,2020,1,Jan,...,1.0,0,1,1,0,1,1,0,0,0
4,SEA_BN_BANDAR_SERI_BEGAWAN,2020-01-05,train,Brunei,Bandar Seri Begawan,4.9031,114.9398,2020,1,Jan,...,1.0,0,1,1,0,1,1,0,0,0


Processed shape: (26304, 60)
Excluded columns: 56


## 9. Feature Dictionary and Quality Report

These tables document the processed schema and the checks that make the dataset ready for the next stage.

In [10]:
feature_rows = []
for column in processed_df.columns:
    if column in id_cols:
        group = 'identifier_or_time'
        modeling_role = 'metadata'
        meaning = 'Identifier, location descriptor, coordinate, temporal split, or calendar field.'
    elif column in target_candidate_cols:
        group = 'target_candidate'
        modeling_role = 'target_candidate'
        meaning = 'Rainfall target candidate. Use sensitivity analysis before selecting a final target.'
    elif column in target_uncertainty_cols:
        group = 'target_uncertainty'
        modeling_role = 'quality_or_weight_feature'
        meaning = 'Feature describing disagreement, calibration uncertainty, or target confidence.'
    elif column in calendar_cols:
        group = 'calendar_encoding'
        modeling_role = 'predictor'
        meaning = 'Cyclic calendar encoding for seasonality.'
    elif column in weather_mean_cols:
        group = 'weather_summary'
        modeling_role = 'predictor'
        meaning = 'Two-source mean meteorological predictor.'
    elif column in weather_uncertainty_cols:
        group = 'weather_source_uncertainty'
        modeling_role = 'predictor'
        meaning = 'Absolute gap between NASA POWER and Open-Meteo for a weather predictor.'
    elif column in history_cols:
        group = 'forecasting_history_feature'
        modeling_role = 'predictor'
        meaning = 'Lagged or rolling rainfall feature computed using previous days only.'
    elif column in imputation_indicator_cols:
        group = 'imputation_indicator'
        modeling_role = 'predictor_quality_flag'
        meaning = 'Binary indicator equal to 1 when the corresponding engineered history feature was imputed.'
    else:
        group = 'other'
        modeling_role = 'review_before_modeling'
        meaning = 'Additional processed field.'

    feature_rows.append({
        'column_name': column,
        'dtype': str(processed_df[column].dtype),
        'feature_group': group,
        'modeling_role': modeling_role,
        'meaning': meaning,
    })

processed_feature_dictionary = pd.DataFrame(feature_rows)

quality_report = pd.DataFrame([
    {'check': 'processed_rows', 'value': int(len(processed_df)), 'expected': int(len(raw_df)), 'passed': len(processed_df) == len(raw_df)},
    {'check': 'processed_columns', 'value': int(processed_df.shape[1]), 'expected': 'slim schema, fewer than 70 columns', 'passed': processed_df.shape[1] < 70},
    {'check': 'processed_entities', 'value': int(processed_df['entity_id'].nunique()), 'expected': int(raw_df['entity_id'].nunique()), 'passed': processed_df['entity_id'].nunique() == raw_df['entity_id'].nunique()},
    {'check': 'processed_missing_cells', 'value': int(processed_df.isna().sum().sum()), 'expected': 0, 'passed': int(processed_df.isna().sum().sum()) == 0},
    {'check': 'negative_target_candidate_rows', 'value': int((processed_df[target_candidate_cols] < 0).any(axis=1).sum()), 'expected': 0, 'passed': int((processed_df[target_candidate_cols] < 0).any(axis=1).sum()) == 0},
    {'check': 'train_rows', 'value': int((processed_df['split'] == 'train').sum()), 'expected': 17532, 'passed': int((processed_df['split'] == 'train').sum()) == 17532},
    {'check': 'validation_rows', 'value': int((processed_df['split'] == 'validation').sum()), 'expected': 4392, 'passed': int((processed_df['split'] == 'validation').sum()) == 4392},
    {'check': 'test_rows', 'value': int((processed_df['split'] == 'test').sum()), 'expected': 4380, 'passed': int((processed_df['split'] == 'test').sum()) == 4380},
])

if not quality_report['passed'].all():
    raise ValueError('Processed quality report has failed checks.')

display(processed_feature_dictionary.head(20))
display(quality_report)
display(excluded_columns_report.head(20))

,column_name,dtype,feature_group,modeling_role,meaning
0,entity_id,object,identifier_or_time,metadata,"Identifier, location descriptor, coordinate, t..."
1,date,datetime64[ns],identifier_or_time,metadata,"Identifier, location descriptor, coordinate, t..."
2,split,object,identifier_or_time,metadata,"Identifier, location descriptor, coordinate, t..."
3,country,object,identifier_or_time,metadata,"Identifier, location descriptor, coordinate, t..."
4,location_name,object,identifier_or_time,metadata,"Identifier, location descriptor, coordinate, t..."
5,canonical_latitude,float64,identifier_or_time,metadata,"Identifier, location descriptor, coordinate, t..."
6,canonical_longitude,float64,identifier_or_time,metadata,"Identifier, location descriptor, coordinate, t..."
7,year,int32,identifier_or_time,metadata,"Identifier, location descriptor, coordinate, t..."
8,month,int32,identifier_or_time,metadata,"Identifier, location descriptor, coordinate, t..."
9,month_name,object,identifier_or_time,metadata,"Identifier, location descriptor, coordinate, t..."


,check,value,expected,passed
0,processed_rows,26304,26304,True
1,processed_columns,60,"slim schema, fewer than 70 columns",True
2,processed_entities,12,12,True
3,processed_missing_cells,0,0,True
4,negative_target_candidate_rows,0,0,True
5,train_rows,17532,17532,True
6,validation_rows,4392,4392,True
7,test_rows,4380,4380,True


,column_name,reason
0,nasa_power_calibrated_to_open_meteo_mm,not_exported_to_keep_processed_dataset_slim; r...
1,nasa_power_precipitation_mm,not_exported_to_keep_processed_dataset_slim; r...
2,nasa_power_relative_humidity_pct,not_exported_to_keep_processed_dataset_slim; r...
3,nasa_power_surface_pressure_kpa,not_exported_to_keep_processed_dataset_slim; r...
4,nasa_power_temp_max_c,not_exported_to_keep_processed_dataset_slim; r...
5,nasa_power_temp_mean_c,not_exported_to_keep_processed_dataset_slim; r...
6,nasa_power_temp_min_c,not_exported_to_keep_processed_dataset_slim; r...
7,nasa_power_wind_speed_ms,not_exported_to_keep_processed_dataset_slim; r...
8,nasa_wet_day,not_exported_to_keep_processed_dataset_slim; r...
9,open_meteo_calibrated_to_nasa_mm,not_exported_to_keep_processed_dataset_slim; r...


## 10. Save Compact Processed Dataset and Reports

To keep the workspace clean, this notebook saves only one data file in `data/processed/`:

- `sea_rainfall_daily_2020_2025_processed_features.csv`

All supporting artifacts go to `reports/03_preprocessing/`:

- bias-correction table;
- imputation report;
- scaling parameters;
- processed feature dictionary;
- quality report;
- split summary;
- README and manifest.

The single processed CSV already contains the `split` column, so separate train/validation/test files are unnecessary.

In [11]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


EXCLUDED_COLUMNS_PATH = TABLE_DIR / '07_excluded_columns_from_slim_dataset.csv'

processed_df.to_csv(PROCESSED_MAIN_PATH, index=False)

# Audit/report tables are intentionally kept outside data/processed.
bias_table.to_csv(BIAS_CORRECTION_PATH, index=False)
imputation_report.to_csv(IMPUTATION_REPORT_PATH, index=False)
scaling_parameters.to_csv(SCALING_PARAMETERS_PATH, index=False)
processed_feature_dictionary.to_csv(FEATURE_DICTIONARY_PATH, index=False)
quality_report.to_csv(QUALITY_REPORT_PATH, index=False)
split_summary.to_csv(SPLIT_SUMMARY_PATH, index=False)
excluded_columns_report.to_csv(EXCLUDED_COLUMNS_PATH, index=False)

readme_lines = [
    '# Preprocessing Report',
    '',
    'This folder contains audit artifacts from `notebooks/03_preprocessing.ipynb`.',
    '',
    '## Data Output',
    f'- Main processed dataset: `data/processed/{PROCESSED_MAIN_PATH.name}`.',
    '- The processed dataset is intentionally slim and already contains the `split` column.',
    '- Separate train/validation/test files are not exported.',
    '- Source-side raw measurements remain in `data/raw/`; the processed table keeps compact target, weather, history, and uncertainty features.',
    '',
    '## Target Candidates',
    '- `target_nasa_power_precipitation_mm`: NASA POWER rainfall.',
    '- `target_open_meteo_precipitation_mm`: Open-Meteo rainfall.',
    '- `target_baseline_two_source_mean_mm`: raw arithmetic mean of the two sources; use as baseline only.',
    '- `target_nasa_reference_consensus_mm`: Open-Meteo calibrated to NASA using train city-month bias, then averaged.',
    '- `target_open_meteo_reference_consensus_mm`: NASA calibrated to Open-Meteo using train city-month bias, then averaged.',
    '',
    '## Why Some Columns Were Dropped',
    '- Raw source rainfall columns duplicate target candidates.',
    '- Raw source-specific weather columns are summarized by two-source means and source-gap features.',
    '- Scaled `z_` columns are not exported to avoid doubling the feature count; scaling parameters are saved in `tables/03_scaling_parameters.csv`.',
    '- Dropped columns are documented in `tables/07_excluded_columns_from_slim_dataset.csv`.',
    '',
    '## Imputation',
    'Raw targets and raw source measurements are not imputed because notebook 03 finds no missing values there.',
    'Only lag/rolling/spell predictors are imputed when missing due to temporal warm-up.',
    'Formula: if missing, fill with median_train(feature | entity_id); fallback median_train(feature).',
    '',
    '## Recommended Use',
    'Do not treat the raw two-source mean as an unquestioned ground truth. Train later models against multiple target candidates and report sensitivity.',
]
README_PATH.write_text('\n'.join(readme_lines), encoding='utf-8')

data_paths = [PROCESSED_MAIN_PATH]
report_paths = [
    BIAS_CORRECTION_PATH,
    IMPUTATION_REPORT_PATH,
    SCALING_PARAMETERS_PATH,
    FEATURE_DICTIONARY_PATH,
    QUALITY_REPORT_PATH,
    SPLIT_SUMMARY_PATH,
    EXCLUDED_COLUMNS_PATH,
    README_PATH,
]

manifest = {
    'dataset_name': 'Processed Southeast Asia daily rainfall dataset 2020-2025',
    'created_by_notebook': 'notebooks/03_preprocessing.ipynb',
    'created_at_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'input_files': [str(RESOLVED_PATH.relative_to(PROJECT_ROOT)), str(REGISTRY_PATH.relative_to(PROJECT_ROOT))],
    'data_output_policy': {
        'processed_data_files': 1,
        'reason': 'Keep data/processed compact. Split labels are stored inside the main processed table.',
        'slim_schema_columns': int(processed_df.shape[1]),
        'excluded_columns_documented_at': str(EXCLUDED_COLUMNS_PATH.relative_to(PROJECT_ROOT)),
    },
    'report_output_directory': str(REPORT_DIR.relative_to(PROJECT_ROOT)),
    'temporal_split': {
        'train': '2020-01-01 to 2023-12-31',
        'validation': '2024-01-01 to 2024-12-31',
        'test': '2025-01-01 to 2025-12-31',
    },
    'target_strategy': {
        'decision': 'Do not blindly average sources as the final target.',
        'reason': 'Notebook 02 found global MAE 5.88 mm/day, RMSE 12.38 mm/day, Pearson 0.42, and strict direct averaging supported for 0/12 cities.',
        'target_candidates': target_candidate_cols,
    },
    'bias_correction_formula': {
        'city_month_bias': 'mean_train(nasa_power_precipitation_mm - open_meteo_precipitation_mm | entity_id, month)',
        'open_meteo_calibrated_to_nasa': 'max(open_meteo_precipitation_mm + city_month_bias, 0)',
        'nasa_power_calibrated_to_open_meteo': 'max(nasa_power_precipitation_mm - city_month_bias, 0)',
    },
    'imputation_policy': {
        'raw_targets_imputed': False,
        'raw_source_measurements_imputed': False,
        'engineered_history_features_imputed': True,
        'formula': 'if missing, fill with median_train(feature | entity_id); fallback median_train(feature)',
    },
    'row_counts': {
        'full': int(len(processed_df)),
        'train': int((processed_df['split'] == 'train').sum()),
        'validation': int((processed_df['split'] == 'validation').sum()),
        'test': int((processed_df['split'] == 'test').sum()),
    },
    'quality_report': quality_report.to_dict(orient='records'),
    'files': [
        {
            'path': str(path.relative_to(PROJECT_ROOT)),
            'role': 'processed_data' if path in data_paths else 'preprocessing_report',
            'sha256': sha256_file(path),
        }
        for path in data_paths + report_paths
    ],
}

MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding='utf-8')

print('Saved processed data:')
for path in data_paths:
    print('-', path.relative_to(PROJECT_ROOT))

print('\nSaved preprocessing reports:')
for path in report_paths + [MANIFEST_PATH]:
    print('-', path.relative_to(PROJECT_ROOT))

Saved processed data:
- data\processed\sea_rainfall_daily_2020_2025_processed_features.csv

Saved preprocessing reports:
- reports\03_preprocessing\tables\01_city_month_bias_correction.csv
- reports\03_preprocessing\tables\02_imputation_report.csv
- reports\03_preprocessing\tables\03_scaling_parameters.csv
- reports\03_preprocessing\tables\04_processed_feature_dictionary.csv
- reports\03_preprocessing\tables\05_processed_quality_report.csv
- reports\03_preprocessing\tables\06_temporal_split_summary.csv
- reports\03_preprocessing\tables\07_excluded_columns_from_slim_dataset.csv
- reports\03_preprocessing\PREPROCESSING_README.md
- reports\03_preprocessing\sea_rainfall_daily_2020_2025_processed_manifest.json
